# <font color="#418FDE" size="6.5" uppercase>**Data Contracts**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Prepare NumPy, pandas, and sparse inputs that satisfy scikit-learn shape requirements. 
- Identify dtype and target-shape issues before fitting estimators. 
- Use dataset containers, feature names, cloning, tags, and diagrams for inspection. 


## **1. Input Formats**

### **1.1. NumPy Array Shapes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_01.jpg?v=1783977045" width="250">



>* Rows are samples; columns are features
>* Use 2D arrays, even for one feature

>* Shape single-feature data as one column
>* Keep prediction columns matching training layout

>* Validate dimensions and avoid transposed feature matrices
>* Keep rows as cases, columns as features



In [ ]:
#@title Python Code - NumPy Array Shapes

# This example teaches scikit-learn array shape contracts.
# Rows are samples and columns are features.
# The output compares wrong and correct shapes.

import numpy as np
from sklearn.linear_model import LinearRegression
import sklearn

# A flat array is ambiguous for scikit-learn estimators.
floor_area_sqft = np.array([500, 650, 800, 950, 1100])
rent_dollars = np.array([1500, 1800, 2100, 2400, 2700])

# Reshape one feature into a two-dimensional feature matrix.
X_one_feature = floor_area_sqft.reshape(-1, 1)

# A single prediction row still needs two dimensions.
new_apartment = np.array([[725]])

# Validate the key shape contract before fitting.
if X_one_feature.ndim != 2:
    raise ValueError("X must be two-dimensional.")

if X_one_feature.shape[0] != rent_dollars.shape[0]:
    raise ValueError("X rows must match target length.")

# Fit one simple model after the shape is explicit.
model = LinearRegression()
model.fit(X_one_feature, rent_dollars)

predicted_rent = model.predict(new_apartment)[0]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Flat input shape: {floor_area_sqft.shape}")
print(f"Correct X shape: {X_one_feature.shape}")
print(f"Target y shape: {rent_dollars.shape}")
print(f"One new sample shape: {new_apartment.shape}")
print(f"Predicted rent for 725 sqft: ${predicted_rent:.0f}")



### **1.2. DataFrame Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_02.jpg?v=1783977040" width="250">



>* Rows are observations; columns are features
>* Keep even one feature two-dimensional

>* DataFrames organize mixed feature types by column
>* Encode features and preserve valid alignment

>* Keep targets separate from predictor columns
>* Maintain row alignment after data operations



In [ ]:
#@title Python Code - DataFrame Inputs

# DataFrames keep feature tables two-dimensional.
# Column selection changes scikit-learn input shape.
# The output compares correct and incorrect inputs.

import pandas as pd
from sklearn.linear_model import LogisticRegression
import sklearn

# Build a tiny in-memory table with named columns.
patients = pd.DataFrame(
    {
        "age_years": [29, 45, 51, 62, 38, 70],
        "cholesterol_mg_dl": [180, 210, 240, 260, 195, 275],
        "diagnosed": [0, 0, 1, 1, 0, 1],
    }
)

# Keep predictors separate from the target column.
feature_columns = ["age_years", "cholesterol_mg_dl"]
X = patients[feature_columns]
y = patients["diagnosed"]

# A double bracket selection preserves a two-dimensional DataFrame.
one_feature_table = patients[["age_years"]]
one_feature_series = patients["age_years"]

# Validate the shapes before fitting an estimator.
if X.ndim != 2 or len(X) != len(y):
    raise ValueError("Features must be a 2D table aligned with the target.")

# Fit one simple estimator using the valid DataFrame input.
model = LogisticRegression(random_state=42)
model.fit(X, y)

print("scikit-learn version:", sklearn.__version__)
print("Full feature DataFrame shape:", X.shape)
print("Single-column DataFrame shape:", one_feature_table.shape)
print("Single-column Series shape:", one_feature_series.shape)
print("Target shape:", y.shape)
print("Model feature names:", list(model.feature_names_in_))



### **1.3. Sparse Matrix Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_03.jpg?v=1783977042" width="250">



>* Sparse data stores only nonzero feature values
>* Keep inputs two-dimensional: samples by features

>* Choose efficient CSR or CSC formats
>* Avoid unsupported sparse inputs causing densification

>* Keep sparse feature columns consistent across datasets
>* Transform raw data into numeric sparse tables



In [ ]:
#@title Python Code - Sparse Matrix Inputs

# Sparse matrices keep mostly zero features compact.
# Scikit-learn still expects two-dimensional feature tables.
# This example checks shape, dtype, and memory.

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
import sklearn

# Small text data creates a sparse document-term matrix.
documents = [
    "cat eats fish",
    "dog eats bone",
    "cat sleeps",
    "dog barks",
]

labels = np.array([0, 1, 0, 1])

# CountVectorizer learns columns and returns a sparse matrix.
vectorizer = CountVectorizer()
X_sparse = vectorizer.fit_transform(documents)

# The sparse matrix must still be samples by features.
if X_sparse.ndim != 2:
    raise ValueError("Feature input must be two-dimensional.")

if X_sparse.shape[0] != labels.shape[0]:
    raise ValueError("Feature rows must match target length.")

# Sparse matrices store only nonzero values and positions.
dense_bytes = X_sparse.toarray().nbytes
sparse_bytes = X_sparse.data.nbytes + X_sparse.indices.nbytes
sparse_bytes = sparse_bytes + X_sparse.indptr.nbytes

# A pandas view helps connect columns to feature names.
feature_names = vectorizer.get_feature_names_out()
preview = pd.DataFrame(
    X_sparse.toarray(), columns=feature_names
).iloc[:3, :5]

# Many estimators accept sparse input directly.
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(X_sparse, labels)

new_documents = ["cat eats", "dog sleeps"]
X_new = vectorizer.transform(new_documents)
predictions = model.predict(X_new)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Sparse shape: {X_sparse.shape[0]} samples x {X_sparse.shape[1]} features")
print(f"Target shape: {labels.shape}")
print(f"Sparse dtype: {X_sparse.dtype}")
print(f"Nonzero entries: {X_sparse.nnz} of {X_sparse.shape[0] * X_sparse.shape[1]}")
print(f"Dense bytes: {dense_bytes}; sparse bytes: {sparse_bytes}")
print("Preview of the dense view:")
print(preview)
print(f"Predictions for new rows: {predictions.tolist()}")



## **2. Shapes and Dtypes**

### **2.1. Feature Matrix Shape**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_01.jpg?v=1783977053" width="250">



>* Rows are samples; columns are features
>* Keep single-feature inputs two-dimensional

>* Avoid flattened or transposed feature matrices
>* Verify rows, targets, and predictor columns

>* Keep feature columns consistent across model use
>* Check shapes after every data transformation



In [ ]:
#@title Python Code - Feature Matrix Shape

# This example checks feature matrix shape.
# Scikit-learn expects two-dimensional feature inputs.
# The output compares wrong and correct shapes.

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import sklearn

# A single feature often starts as a one-dimensional sequence.
temperature_c = np.array([12.0, 14.0, 16.0, 18.0, 20.0])
energy_kwh = np.array([30.0, 34.0, 39.0, 43.0, 48.0])

# This shape is wrong for a scikit-learn feature matrix.
wrong_feature_matrix = temperature_c

# Reshape keeps one sample per row and one feature column.
correct_feature_matrix = temperature_c.reshape(-1, 1)

# A pandas double bracket selection also preserves two dimensions.
weather_table = pd.DataFrame({"temperature_c": temperature_c})
pandas_feature_matrix = weather_table[["temperature_c"]]

# Fit only after confirming rows match the target length.
row_count_matches = correct_feature_matrix.shape[0] == energy_kwh.shape[0]
model = LinearRegression()
model.fit(correct_feature_matrix, energy_kwh)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Wrong NumPy shape: {wrong_feature_matrix.shape}")
print(f"Correct NumPy shape: {correct_feature_matrix.shape}")
print(f"Pandas double-bracket shape: {pandas_feature_matrix.shape}")
print(f"Rows match target length: {row_count_matches}")
print(f"Prediction for 22 degrees C: {model.predict([[22.0]])[0]:.1f} kWh")



### **2.2. Target Shape Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_02.jpg?v=1783977055" width="250">



>* Match each feature row to one target
>* Check counts after preprocessing changes

>* Single-output targets should usually be one-dimensional
>* Check for accidental pandas extra dimensions

>* Two-dimensional targets can be intentional
>* Verify shape matches task and estimator



In [ ]:
#@title Python Code - Target Shape Checks

# This script checks target shapes before fitting.
# It compares valid and invalid target arrays.
# The output shows beginner-friendly shape decisions.

import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import sklearn

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Build common target shapes for comparison.
y_good = y
y_column = y.reshape(-1, 1)
y_too_short = y[:-1]
y_multi_output = np.column_stack((y == 0, y == 1))

# Store each target with its intended meaning.
targets = [
    ("1D labels", y_good, "single-output classification"),
    ("2D one-column", y_column, "usually flatten for single-output"),
    ("too short", y_too_short, "row-count mismatch"),
    ("2D two-column", y_multi_output, "intentional multi-output style"),
]

print("scikit-learn version:", sklearn.__version__)
print("Feature matrix shape:", X.shape)

# Check row alignment and target dimensionality.
for name, target, meaning in targets:
    rows_match = len(target) == X.shape[0]
    shape_text = str(target.shape)
    print(name, "shape", shape_text, "rows match:", rows_match)

# Flatten the accidental one-column target before fitting.
y_flattened = y_column.ravel()
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X, y_flattened)

# Confirm the corrected target works with the estimator.
print("Flattened target shape:", y_flattened.shape)
print("Model fit after flattening:", hasattr(model, "classes_"))



### **2.3. Dtype Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_03.jpg?v=1783977057" width="250">



>* Check feature dtypes before fitting estimators
>* Transform nonnumeric columns into usable representations

>* Separate true numbers from numeric labels
>* Match target dtypes to prediction task

>* Check dtypes throughout data preparation
>* Ensure values fit estimator expectations



In [ ]:
#@title Python Code - Dtype Checks

# This example checks dtypes before fitting.
# It separates numeric measurements from labels.
# It shows target-shape and dtype readiness.

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Build a tiny dataset with common dtype problems.
houses = pd.DataFrame(
    {
        "area_sq_m": [70, 85, 120, 95],
        "price_text": ["$210,000", "$255,000", "$360,000", "$285,000"],
        "postal_code": [10101, 10102, 10101, 10103],
    }
)

# Inspect how pandas currently stores each column.
print("Feature dtypes before cleaning:")
print(houses.dtypes.to_string())

# Convert the true measurement column from text to numbers.
clean_price = houses["price_text"].str.replace("$", "", regex=False)
clean_price = clean_price.str.replace(",", "", regex=False)
houses["price_usd"] = pd.to_numeric(clean_price)

# Keep only meaningful numeric measurements for this regression example.
feature_matrix = houses[["area_sq_m"]].to_numpy(dtype=np.float64)
target_vector = houses["price_usd"].to_numpy(dtype=np.float64)

# Validate the estimator contract before fitting.
print("Feature matrix shape:", feature_matrix.shape)
print("Target vector shape:", target_vector.shape)
print("Feature dtype:", feature_matrix.dtype)
print("Target dtype:", target_vector.dtype)

# Fit only after dtype and target-shape checks pass.
model = LinearRegression()
model.fit(feature_matrix, target_vector)

# Show why postal codes should not be treated as measurements.
print("Postal code dtype:", houses["postal_code"].dtype)
print("Postal code meaning: label, not continuous measurement")
print("Ready for regression fit:", target_vector.ndim == 1)
print("Model slope dollars per square meter:", round(model.coef_[0], 2))



## **3. Inspection Utilities**

### **3.1. Bunch Data Containers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_01.jpg?v=1783977047" width="250">



>* Bunch stores data with helpful metadata
>* Inspect arrays before fitting estimators

>* Check feature data against target values
>* Organize dataset parts for easier inspection

>* Connect arrays to meaningful feature names
>* Inspect models with traceable metadata



In [ ]:
#@title Python Code - Bunch Data Containers

# Inspect a scikit-learn Bunch container.
# Connect arrays with names and metadata.
# Print compact checks before modeling.

from sklearn.datasets import load_iris

# Load a packaged dataset returned as a Bunch.
iris_bunch = load_iris()

# A Bunch supports dictionary keys and attribute access.
print("Container type:", type(iris_bunch).__name__)
print("Selected keys:", list(iris_bunch.keys())[:5])

# Inspect the estimator contract pieces before fitting.
feature_matrix = iris_bunch.data
target_vector = iris_bunch.target
print("data shape:", feature_matrix.shape)
print("target shape:", target_vector.shape)

# Feature and target names explain array columns and labels.
print("first feature names:", list(iris_bunch.feature_names[:3]))
print("target names:", list(iris_bunch.target_names))

# Validate the most important shape relationship.
rows_match = feature_matrix.shape[0] == target_vector.shape[0]
print("same number of samples:", rows_match)

# Attribute access and dictionary access point to the same object.
same_data_object = iris_bunch.data is iris_bunch["data"]
print("attribute equals key access:", same_data_object)



### **3.2. Cloning Estimators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_02.jpg?v=1783977049" width="250">



>* Copies estimator settings, not learned results
>* Creates a clean model for refitting

>* Fresh clones prevent validation data leakage
>* Pipelines and grids compare clean candidates

>* Separate model settings from learned results
>* Clone workflows for independent, trustworthy inspection



In [ ]:
#@title Python Code - Cloning Estimators

# This example shows how estimator cloning works.
# Clones keep settings but forget learned attributes.
# The output compares fitted and cloned estimators.

import sklearn
from sklearn.base import clone
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Validate the basic data contract before fitting.
if X.ndim != 2 or y.ndim != 1:
    raise ValueError("Expected two-dimensional features and one-dimensional target.")

if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target length.")

# Configure one estimator, then fit it once.
original_model = LogisticRegression(max_iter=200, random_state=42)
original_model.fit(X, y)

# Clone copies configuration but not fitted attributes.
cloned_model = clone(original_model)

# Inspect configuration and learned-state differences.
same_settings = original_model.get_params() == cloned_model.get_params()
original_is_fitted = hasattr(original_model, "classes_")
clone_is_fitted = hasattr(cloned_model, "classes_")

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Same hyperparameter settings: {same_settings}")
print(f"Original has learned classes_: {original_is_fitted}")
print(f"Clone has learned classes_: {clone_is_fitted}")
print(f"Original and clone are same object: {original_model is cloned_model}")



### **3.3. Feature Name Inspection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_03.jpg?v=1783977051" width="250">



>* Check how estimators track input feature names
>* Catch column mismatches before modeling errors

>* Check scoring data matches training features
>* Trace transformed columns into model inputs

>* Use feature names for shared model understanding
>* Verify data contracts and expected inputs



In [ ]:
#@title Python Code - Feature Name Inspection

# Inspect feature names stored by fitted estimators.
# Compare pandas names with plain array inputs.
# Confirm column order before making predictions.

import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression

# Build a tiny labeled dataset with meaningful column names.
training_data = pd.DataFrame(
    {
        "age_years": [22, 25, 47, 52, 46, 56, 28, 31],
        "income_k": [35, 40, 82, 90, 78, 95, 45, 50],
        "debt_ratio": [0.20, 0.25, 0.55, 0.60, 0.50, 0.65, 0.30, 0.35],
    }
)

# The target is small and shaped as one value per row.
target = [0, 0, 1, 1, 1, 1, 0, 0]

# Fit one simple estimator on the named pandas columns.
model = LogisticRegression(random_state=42)
model.fit(training_data, target)

# Inspect the names scikit-learn recorded during fitting.
expected_names = list(model.feature_names_in_)
print("scikit-learn version:", sklearn.__version__)
print("Names stored after fit:", expected_names)

# Create new data with the same columns in the wrong order.
scoring_data = training_data[["debt_ratio", "age_years", "income_k"]].head(2)
current_names = list(scoring_data.columns)
print("Incoming column order:", current_names)

# Reorder by name before prediction to satisfy the data contract.
names_match = current_names == expected_names
safe_scoring_data = scoring_data[expected_names]
predictions = model.predict(safe_scoring_data)
print("Column order already matched:", names_match)
print("Predictions after name-based reorder:", predictions.tolist())



# <font color="#418FDE" size="6.5" uppercase>**Data Contracts**</font>


In this lecture, you learned to:
- Prepare NumPy, pandas, and sparse inputs that satisfy scikit-learn shape requirements. 
- Identify dtype and target-shape issues before fitting estimators. 
- Use dataset containers, feature names, cloning, tags, and diagrams for inspection. 

In the next Module (Module 3), we will go over 'Datasets'